In [ ]:

from Model import Model
import pandas as pd
from datasets import Dataset
import os
from dotenv import load_dotenv
from BaselineModel import BaselineModel
from constant import *
from SimpleOutputLabelConverter import SimpleOutputLabelConverter

# df = pd.read_csv("../cache/output/detect_output_metrics.csv")
# df.insert(6, "mcc", None)
# df.insert(7, "auc", None)
# df.to_csv("../cache/output/detect_output_metrics.csv", index=False)
df = pd.read_csv("../cache/output/detect_output_metrics_bk_latest.csv")
df = df[df["metric"] == "yes"]
for uri, dataset in zip(df["model"], df["dataset"]):
    merged_file_name = f"../cache/output/merged/merged_detect_{uri.split('/')[-1]}.csv"
    if os.path.exists(merged_file_name):
        detect_df = pd.read_csv(f'../data/{dataset}_detect_test.csv')
        detect_dataset = Dataset.from_pandas(detect_df)
        model = Model("detect", "pretrained-liu-detector", SimpleOutputLabelConverter({'yes', 'no'}, DEFAULT_DETECTION_CLASS), 100_000)
        merged_df = pd.read_csv(merged_file_name)
        merged_df.set_index('id', inplace=True)
        mapping = merged_df["label_pred"].to_dict()
        labels_prediction = detect_df["id"].map(mapping)

        # Check for missing mappings
        if labels_prediction.isnull().any():
            missing_ids = detect_df.loc[labels_prediction.isnull(), "id"].unique()
            raise ValueError(f"Missing mapping for ids: {missing_ids}")
        labels_prediction = labels_prediction.tolist()
        model.inject_mcc_auc(detect_dataset, dataset, labels_prediction,labels_prediction, uri, None )
    else:
        print(uri, dataset)
        print(merged_file_name)